In [1]:
import escher
import cobra
import pandas as pd
import itertools

In [2]:
# Load the models
amac_model = cobra.io.read_sbml_model("../model.xml")
ecoli_model = cobra.io.read_sbml_model("/Users/helenscott/Documents/PhD/Segre-lab/GEM-repos/ecoli/iJO1366.xml")

In [3]:
# Save the paths to the maps
amac_map_path = "../escher/MIT1002_TCA_escher-map.json"
ecoli_map_path = "/Users/helenscott/Documents/PhD/Segre-lab/GEM-repos/ecoli/iJO1366.TCA Only.json"

In [4]:
# Define media without carbon or nitrogen sources
amac_basal_media = {
    "EX_cpd00058_e0": 1000,  # Cu2+_e0
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00971_e0": 1000,  # Na+_e0
    "EX_cpd00063_e0": 1000,  # Ca2+_e0
    "EX_cpd00048_e0": 1000,  # Sulfate_e0
    "EX_cpd10516_e0": 1000,  # fe3_e0
    "EX_cpd00254_e0": 1000,  # Mg_e0
    "EX_cpd00009_e0": 1000,  # Phosphate_e0
    "EX_cpd00205_e0": 1000,  # K+_e0
    "EX_cpd00099_e0": 1000,  # Cl-_e0
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
    "EX_cpd00001_e0": 1000,  # H2O_e0
    "EX_cpd00034_e0": 1000,  # Zn2+_e0
    "EX_cpd00149_e0": 1000,  # Co2+_e0
}
ecoli_basal_media = {
    'EX_co2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_h_e': 1000.0,
    'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    'EX_mobd_e': 1000.0,
    'EX_na1_e': 1000.0,
    'EX_ca2_e': 1000.0,
    'EX_cbl1_e': 0.01,
    'EX_ni2_e': 1000.0,
    'EX_o2_e': 1000.0,
    'EX_cl_e': 1000.0,
    'EX_pi_e': 1000.0,
    'EX_zn2_e': 1000.0,
    'EX_sel_e': 1000.0,
    'EX_slnt_e': 1000.0,
    'EX_so4_e': 1000.0,
    'EX_tungs_e': 1000.0,
    'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0
 }

In [5]:
# Define additions to the minimal media
c_sources = {"Glucose": {"ModelSEED": "EX_cpd00027_e0", "BiGG": "EX_glc__D_e"},
             "Acetate": {"ModelSEED": "EX_cpd00029_e0", "BiGG": "EX_ac_e"}}
n_sources = {"Ammonia": {"ModelSEED": "EX_cpd00013_e0", "BiGG": "EX_nh4_e"},
             "Nitrate": {"ModelSEED": "EX_cpd00209_e0", "BiGG": "EX_no3_e"}}

In [6]:
# Make list of all reactions to maximize
modelseed_tca_rxns = [
    "rxn00256_c0",
    "rxn00974_c0",
    "rxn01388_c0",
    "rxn01387_c0",
    "rxn00199_c0",
    "rxn00336_c0",
    "rxn00330_c0",
    "rxn00441_c0",
    "rxn02376_c0",
    "rxn01872_c0",
    "rxn08094_c0",
    "rxn00285_c0",
    "rxn00288_c0",
    "rxn10126_c0",
    "rxn00799_c0",
    "rxn00248_c0"
]
bigg_tca_rxns = [
    "ACONTa",
    "ACONTb",
    "ICDHyr",
    "AKGDH",
    "SUCOAS",
    "FRD2",
    "SUCDi",
    "FUM",
    "MOX",
    "MDH2",
    "MDH",
    "MDH3",
    "CS",
    "CITL",
    "MALS",
    "ICL"
]

# Optimize for Biomass (Check Fluxes When Growing)

## For Amac

In [8]:
# Make a list to store results
amac_growth_results = []
c_names = list(c_sources.keys())
n_names = list(n_sources.keys())

# Iterate over all non-empty subsets of carbon and nitrogen sources
for c_k in range(1, len(c_names) + 1):
    for c_subset in itertools.combinations(c_names, c_k):
        for n_k in range(1, len(n_names) + 1):
            for n_subset in itertools.combinations(n_names, n_k):
                c_label = "+".join(c_subset)
                n_label = "+".join(n_subset)
                print(f'Testing AMAC with C source(s): {c_label} and N source(s): {n_label}')
                # Make a medium (copy so basal isn't mutated)
                amac_medium = amac_basal_media.copy()
                # Add carbon sources (use same per-source bounds as before)
                for cname in c_subset:
                    cid = c_sources[cname]["ModelSEED"]
                    if cname == "Glucose":
                        amac_medium[cid] = 10
                    else:
                        amac_medium[cid] = 30
                # Add nitrogen sources (unlimited)
                for nname in n_subset:
                    nid = n_sources[nname]["ModelSEED"]
                    amac_medium[nid] = 1000
                # Set medium and objective, then optimize
                amac_model.medium = amac_medium
                amac_solution = amac_model.optimize()
                # Collect TCA fluxes
                rxn_fluxes = {rxn: amac_solution.fluxes[rxn]
                              for rxn in modelseed_tca_rxns
                              if rxn in amac_solution.fluxes}
                rxn_fluxes["bio1_biomass"] = amac_solution.objective_value
                amac_growth_results.append({
                    "C_source": c_label,
                    "N_source": n_label,
                    "fluxes": rxn_fluxes,
                })


# Convert to DataFrame for easier viewing
amac_growth_df = pd.DataFrame(amac_growth_results)
amac_growth_df

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


Testing AMAC with C source(s): Glucose and N source(s): Ammonia
Testing AMAC with C source(s): Glucose and N source(s): Nitrate


Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the mos

Testing AMAC with C source(s): Glucose and N source(s): Ammonia+Nitrate
Testing AMAC with C source(s): Acetate and N source(s): Ammonia
Testing AMAC with C source(s): Acetate and N source(s): Nitrate
Testing AMAC with C source(s): Acetate and N source(s): Ammonia+Nitrate
Testing AMAC with C source(s): Glucose+Acetate and N source(s): Ammonia
Testing AMAC with C source(s): Glucose+Acetate and N source(s): Nitrate
Testing AMAC with C source(s): Glucose+Acetate and N source(s): Ammonia+Nitrate


,C_source,N_source,fluxes
0,Glucose,Ammonia,"{'rxn00256_c0': 1.441541959429365, 'rxn00974_c..."
1,Glucose,Nitrate,"{'rxn00256_c0': 2.915341013993146, 'rxn00974_c..."
2,Glucose,Ammonia+Nitrate,"{'rxn00256_c0': 1.4415419594292744, 'rxn00974_..."
3,Acetate,Ammonia,"{'rxn00256_c0': 11.376140138931225, 'rxn00974_..."
4,Acetate,Nitrate,"{'rxn00256_c0': 14.99999999999987, 'rxn00974_c..."
5,Acetate,Ammonia+Nitrate,"{'rxn00256_c0': 11.932749347344862, 'rxn00974_..."
6,Glucose+Acetate,Ammonia,"{'rxn00256_c0': 11.708580505161153, 'rxn00974_..."
7,Glucose+Acetate,Nitrate,"{'rxn00256_c0': 8.520827810772923, 'rxn00974_c..."
8,Glucose+Acetate,Ammonia+Nitrate,"{'rxn00256_c0': 11.708580505161109, 'rxn00974_..."


### Map for Gluocse + Ammonia

In [10]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/557993070.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 1.441541959429365, 'rxn00974_c0': 1.4415419594293652, 'rxn01388_c0': -1.…

### Map for Glucose + Nitrate

In [11]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/1879366724.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 2.915341013993146, 'rxn00974_c0': 2.9153410139931464, 'rxn01388_c0': -2.…

### Map for Glucose + Ammonia + Nitrate

In [12]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/3046197503.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 1.4415419594292744, 'rxn00974_c0': 1.4415419594292744, 'rxn01388_c0': -1…

### Map for Acetate and Ammonia

In [13]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/3190541284.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 11.376140138931225, 'rxn00974_c0': 11.376140138931227, 'rxn01388_c0': -1…

### Map for Acetate and Nitrate

In [14]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/148143941.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 14.99999999999987, 'rxn00974_c0': 14.999999999999872, 'rxn01388_c0': -14…

### Map for Acetate + Ammonia + Nitrate

In [15]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/1535404499.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 11.932749347344862, 'rxn00974_c0': 11.932749347344863, 'rxn01388_c0': -1…

### Map for Glucose + Acetate + Ammonia

In [16]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose+Acetate"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/2593759012.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose+Acetate"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 11.708580505161153, 'rxn00974_c0': 11.708580505161155, 'rxn01388_c0': -1…

### Map for Glucose + Acetate + Nitrate

In [17]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose+Acetate"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/3128731258.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose+Acetate"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 8.520827810772923, 'rxn00974_c0': 8.520827810772925, 'rxn01388_c0': -8.5…

### Map for Glucose + Acetate + Ammonia + Nitrate

In [18]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose+Acetate"][amac_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/4018713813.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose+Acetate"][amac_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 11.708580505161109, 'rxn00974_c0': 11.708580505161109, 'rxn01388_c0': -1…

## For E. coli

In [12]:
print(ecoli_model.objective)

Maximize
1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1


In [19]:
# Make a list to store results
ecoli_growth_results = []

# Iterate over all non-empty subsets of carbon and nitrogen sources
for c_k in range(1, len(c_names) + 1):
    for c_subset in itertools.combinations(c_names, c_k):
        for n_k in range(1, len(n_names) + 1):
            for n_subset in itertools.combinations(n_names, n_k):
                c_label = "+".join(c_subset)
                n_label = "+".join(n_subset)
                # Debugging
                print(f'Testing E. coli with C source: {c_label} and N source: {n_label}')
                # Make a medium
                ecoli_medium = ecoli_basal_media.copy()
                # Add carbon sources (use same per-source bounds as before)
                for cname in c_subset:
                    cid = c_sources[cname]["BiGG"]
                    if cname == "Glucose":
                        ecoli_medium[cid] = 10
                    else:
                        ecoli_medium[cid] = 30
                # Add nitrogen sources (unlimited)
                for nname in n_subset:
                    nid = n_sources[nname]["BiGG"]
                    ecoli_medium[nid] = 1000
                # Set the medium
                ecoli_model.medium = ecoli_medium
                # Optimize for Biomass (Check Fluxes When Growing)
                ecoli_solution = ecoli_model.optimize()
                # Get the fluxes through the TCA cycle reactions (skip reactions that aren't in the model)
                rxn_fluxes = {rxn: ecoli_solution.fluxes[rxn]
                              for rxn in bigg_tca_rxns
                              if rxn in ecoli_solution.fluxes}
                # Add the biomass flux to the dict
                rxn_fluxes["BIOMASS_Ec_iJO1366_core_53p95M"] = ecoli_solution.objective_value
                # Store the results
                ecoli_growth_results.append({
                    "C_source": c_label,
                    "N_source": n_label,
                    "fluxes": rxn_fluxes,
                })
# Convert to DataFrame for easier viewing
ecoli_growth_df = pd.DataFrame(ecoli_growth_results)
ecoli_growth_df

Testing E. coli with C source: Glucose and N source: Ammonia
Testing E. coli with C source: Glucose and N source: Nitrate
Testing E. coli with C source: Glucose and N source: Ammonia+Nitrate
Testing E. coli with C source: Acetate and N source: Ammonia
Testing E. coli with C source: Acetate and N source: Nitrate
Testing E. coli with C source: Acetate and N source: Ammonia+Nitrate
Testing E. coli with C source: Glucose+Acetate and N source: Ammonia
Testing E. coli with C source: Glucose+Acetate and N source: Nitrate
Testing E. coli with C source: Glucose+Acetate and N source: Ammonia+Nitrate


,C_source,N_source,fluxes
0,Glucose,Ammonia,"{'ACONTa': 4.857776644226277, 'ACONTb': 4.8577..."
1,Glucose,Nitrate,"{'ACONTa': 7.452338689463803, 'ACONTb': 7.4523..."
2,Glucose,Ammonia+Nitrate,"{'ACONTa': 4.857776644226272, 'ACONTb': 4.8577..."
3,Acetate,Ammonia,"{'ACONTa': 19.270035975298423, 'ACONTb': 19.27..."
4,Acetate,Nitrate,"{'ACONTa': 20.72967553157014, 'ACONTb': 20.729..."
5,Acetate,Ammonia+Nitrate,"{'ACONTa': 19.27003597529529, 'ACONTb': 19.270..."
6,Glucose+Acetate,Ammonia,"{'ACONTa': 24.932195783058585, 'ACONTb': 24.93..."
7,Glucose+Acetate,Nitrate,"{'ACONTa': 28.377798347837903, 'ACONTb': 28.37..."
8,Glucose+Acetate,Ammonia+Nitrate,"{'ACONTa': 24.932195783058525, 'ACONTb': 24.93..."


### Map for Glucose and Ammonia

In [20]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/620635751.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 4.857776644226277, 'ACONTb': 4.857776644226277, 'ICDHyr': 4.857776644226276, …

### Map for Glucose and Nitrate

In [21]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/880266042.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 7.452338689463803, 'ACONTb': 7.452338689463802, 'ICDHyr': 7.4523386894638035,…

### Map for Glucose + Ammonia + Nitrate

In [22]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/3475103320.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 4.857776644226272, 'ACONTb': 4.857776644226271, 'ICDHyr': 4.857776644226271, …

### Map for Acetate and Ammonia

In [23]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/423020520.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 19.270035975298423, 'ACONTb': 19.27003597529842, 'ICDHyr': 13.235702381148846…

### Map for Acetate and Nitrate

In [24]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/1460944637.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 20.72967553157014, 'ACONTb': 20.729675531570138, 'ICDHyr': 15.516216265562374…

### Map for Acetate + Ammonia + Nitrate

In [26]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/2158028210.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 19.27003597529529, 'ACONTb': 19.270035975295286, 'ICDHyr': 13.23570238114431,…

### Map for Glucose + Acetate + Ammonia

In [27]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose+Acetate"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/3248561471.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose+Acetate"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 24.932195783058585, 'ACONTb': 24.93219578305858, 'ICDHyr': 23.685536716653015…

### Map for Glucose + Acetate + Nitrate

In [28]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose+Acetate"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/1759238242.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose+Acetate"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 28.377798347837903, 'ACONTb': 28.3777983478379, 'ICDHyr': 26.181041549875065,…

### Map for Glucose + Acetate + Ammonia + Nitrate

In [29]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose+Acetate"][ecoli_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_71062/2510059802.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose+Acetate"][ecoli_growth_df["N_source"] == "Ammonia+Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 24.932195783058525, 'ACONTb': 24.93219578305852, 'ICDHyr': 23.685536716652916…

# Optimize for TCA Cylce Reactions (Check for Blockages)

## For Amac

In [18]:
# Make a list to store results
amac_blockage_results = []
# Get every combination of C and N source
for c_name, c_id_dict in c_sources.items():
    for n_name, n_id_dict in n_sources.items():
        # Debugging
        print(f'Testing AMAC with C source: {c_name} ({c_id_dict["ModelSEED"]}) and N source: {n_name} ({n_id_dict["ModelSEED"]})')
        # Make a medium
        amac_medium = amac_basal_media.copy()
        # Add a limiting amount of carbon source
        # But use different bounds for glucose and acetate so the number of carbon atoms is the same
        if c_name == "Glucose":
            amac_medium[c_id_dict["ModelSEED"]] = 10
        else:
            amac_medium[c_id_dict["ModelSEED"]] = 30
        amac_medium[n_id_dict["ModelSEED"]] = 1000
        # Set the medium
        amac_model.medium = amac_medium
        # Make a dictionary to store individual reaction fluxes
        rxn_fluxes = {}
        # Optimize for each of the TCA cycle reactions individually
        for rxn_id in modelseed_tca_rxns:
            # Skip if the reaction is not in the model
            if rxn_id not in [r.id for r in amac_model.reactions]:
                continue
            # Set the objective to the current reaction
            amac_model.objective = rxn_id
            # Debugging
            print(f'  Optimizing for reaction: {rxn_id}')
            # Optimize
            amac_solution = amac_model.optimize()
            # Store the flux
            rxn_fluxes[rxn_id] = amac_solution.objective_value
        # Store the results
        amac_blockage_results.append({
            "C_source": c_name,
            "N_source": n_name,
            "fluxes": rxn_fluxes,
        })
# Convert to DataFrame for easier viewing
amac_blockage_df = pd.DataFrame(amac_blockage_results)
amac_blockage_df

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


Testing AMAC with C source: Glucose (EX_cpd00027_e0) and N source: Ammonia (EX_cpd00013_e0)
  Optimizing for reaction: rxn00256_c0
  Optimizing for reaction: rxn00974_c0
  Optimizing for reaction: rxn01388_c0
  Optimizing for reaction: rxn01387_c0
  Optimizing for reaction: rxn00199_c0
  Optimizing for reaction: rxn00336_c0
  Optimizing for reaction: rxn00330_c0
  Optimizing for reaction: rxn00441_c0
  Optimizing for reaction: rxn02376_c0
  Optimizing for reaction: rxn01872_c0


Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


  Optimizing for reaction: rxn00285_c0
  Optimizing for reaction: rxn00288_c0
  Optimizing for reaction: rxn10126_c0
  Optimizing for reaction: rxn00799_c0
  Optimizing for reaction: rxn00248_c0
Testing AMAC with C source: Glucose (EX_cpd00027_e0) and N source: Nitrate (EX_cpd00209_e0)
  Optimizing for reaction: rxn00256_c0
  Optimizing for reaction: rxn00974_c0
  Optimizing for reaction: rxn01388_c0
  Optimizing for reaction: rxn01387_c0
  Optimizing for reaction: rxn00199_c0
  Optimizing for reaction: rxn00336_c0
  Optimizing for reaction: rxn00330_c0
  Optimizing for reaction: rxn00441_c0
  Optimizing for reaction: rxn02376_c0
  Optimizing for reaction: rxn01872_c0
  Optimizing for reaction: rxn00285_c0


Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


  Optimizing for reaction: rxn00288_c0
  Optimizing for reaction: rxn10126_c0
  Optimizing for reaction: rxn00799_c0
  Optimizing for reaction: rxn00248_c0
Testing AMAC with C source: Acetate (EX_cpd00029_e0) and N source: Ammonia (EX_cpd00013_e0)
  Optimizing for reaction: rxn00256_c0
  Optimizing for reaction: rxn00974_c0
  Optimizing for reaction: rxn01388_c0
  Optimizing for reaction: rxn01387_c0
  Optimizing for reaction: rxn00199_c0
  Optimizing for reaction: rxn00336_c0
  Optimizing for reaction: rxn00330_c0
  Optimizing for reaction: rxn00441_c0


Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


  Optimizing for reaction: rxn02376_c0
  Optimizing for reaction: rxn01872_c0
  Optimizing for reaction: rxn00285_c0
  Optimizing for reaction: rxn00288_c0
  Optimizing for reaction: rxn10126_c0
  Optimizing for reaction: rxn00799_c0
  Optimizing for reaction: rxn00248_c0
Testing AMAC with C source: Acetate (EX_cpd00029_e0) and N source: Nitrate (EX_cpd00209_e0)
  Optimizing for reaction: rxn00256_c0
  Optimizing for reaction: rxn00974_c0
  Optimizing for reaction: rxn01388_c0
  Optimizing for reaction: rxn01387_c0
  Optimizing for reaction: rxn00199_c0
  Optimizing for reaction: rxn00336_c0
  Optimizing for reaction: rxn00330_c0
  Optimizing for reaction: rxn00441_c0
  Optimizing for reaction: rxn02376_c0
  Optimizing for reaction: rxn01872_c0
  Optimizing for reaction: rxn00285_c0
  Optimizing for reaction: rxn00288_c0
  Optimizing for reaction: rxn10126_c0
  Optimizing for reaction: rxn00799_c0
  Optimizing for reaction: rxn00248_c0


,C_source,N_source,fluxes
0,Glucose,Ammonia,"{'rxn00256_c0': 30.588235294117624, 'rxn00974_..."
1,Glucose,Nitrate,"{'rxn00256_c0': 40.0, 'rxn00974_c0': 39.999999..."
2,Acetate,Ammonia,"{'rxn00256_c0': 36.47058823529608, 'rxn00974_c..."
3,Acetate,Nitrate,"{'rxn00256_c0': 30.000000000003265, 'rxn00974_..."


### Map for Glucose and Ammonia

In [19]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Glucose"][amac_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/1300241741.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Glucose"][amac_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 30.588235294117624, 'rxn00974_c0': 30.588235294117663, 'rxn01388_c0': -1…

### Map for Glucose and Nitrate

In [20]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Glucose"][amac_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/2685808883.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Glucose"][amac_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 40.0, 'rxn00974_c0': 39.99999999999843, 'rxn01388_c0': 4.542305946328844…

### Map for Acetate and Ammonia

In [21]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Acetate"][amac_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/2339238323.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Acetate"][amac_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 36.47058823529608, 'rxn00974_c0': 36.47058823529719, 'rxn01388_c0': 2.61…

### Map for Acetate and Nitrate

In [22]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Acetate"][amac_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/1710809783.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_blockage_df[amac_blockage_df["C_source"] == "Acetate"][amac_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 30.000000000003265, 'rxn00974_c0': 30.000000000001855, 'rxn01388_c0': 4.…

## For E. coli

In [23]:
# Make a list to store results
ecoli_blockage_results = []
# Get every combination of C and N source
for c_name, c_id_dict in c_sources.items():
    for n_name, n_id_dict in n_sources.items():
        # Debugging
        print(f'Testing E. coli with C source: {c_name} ({c_id_dict["BiGG"]}) and N source: {n_name} ({n_id_dict["BiGG"]})')
        # Make a medium
        ecoli_medium = ecoli_basal_media.copy()
        # Add a limiting amount of carbon source
        # But use different bounds for glucose and acetate so the number of carbon atoms is the same
        if c_name == "Glucose":
            ecoli_medium[c_id_dict["BiGG"]] = 10
        else:
            ecoli_medium[c_id_dict["BiGG"]] = 30
        # Add an unlimited amount of nitrogen source
        ecoli_medium[n_id_dict["BiGG"]] = 1000
        # Set the medium
        ecoli_model.medium = ecoli_medium
        # Make a dictionary to store individual reaction fluxes
        rxn_fluxes = {}
        # Optimize for each of the TCA cycle reactions individually
        for rxn_id in bigg_tca_rxns:
            # Skip if the reaction is not in the model
            if rxn_id not in [r.id for r in ecoli_model.reactions]:
                continue
            # Set the objective to the current reaction
            ecoli_model.objective = rxn_id
            # Debugging
            print(f'  Optimizing for reaction: {rxn_id}')
            # Optimize
            ecoli_solution = ecoli_model.optimize()
            # Store the flux
            rxn_fluxes[rxn_id] = ecoli_solution.objective_value
        # Store the results
        ecoli_blockage_results.append({
            "C_source": c_name,
            "N_source": n_name,
            "fluxes": rxn_fluxes,
        })
# Convert to DataFrame for easier viewing
ecoli_blockage_df = pd.DataFrame(ecoli_blockage_results)
ecoli_blockage_df

Testing E. coli with C source: Glucose (EX_glc__D_e) and N source: Ammonia (EX_nh4_e)
  Optimizing for reaction: ACONTa
  Optimizing for reaction: ACONTb
  Optimizing for reaction: ICDHyr
  Optimizing for reaction: AKGDH
  Optimizing for reaction: SUCOAS
  Optimizing for reaction: FRD2
  Optimizing for reaction: SUCDi
  Optimizing for reaction: FUM
  Optimizing for reaction: MOX
  Optimizing for reaction: MDH2
  Optimizing for reaction: MDH
  Optimizing for reaction: MDH3
  Optimizing for reaction: CS
  Optimizing for reaction: CITL
  Optimizing for reaction: MALS
  Optimizing for reaction: ICL
Testing E. coli with C source: Glucose (EX_glc__D_e) and N source: Nitrate (EX_no3_e)
  Optimizing for reaction: ACONTa
  Optimizing for reaction: ACONTb
  Optimizing for reaction: ICDHyr
  Optimizing for reaction: AKGDH
  Optimizing for reaction: SUCOAS
  Optimizing for reaction: FRD2
  Optimizing for reaction: SUCDi
  Optimizing for reaction: FUM
  Optimizing for reaction: MOX
  Optimizing for

,C_source,N_source,fluxes
0,Glucose,Ammonia,"{'ACONTa': 93.8266666666663, 'ACONTb': 93.8266..."
1,Glucose,Nitrate,"{'ACONTa': 93.82666666666738, 'ACONTb': 93.826..."
2,Acetate,Ammonia,"{'ACONTa': 95.16000000000092, 'ACONTb': 95.160..."
3,Acetate,Nitrate,"{'ACONTa': 95.15999999999853, 'ACONTb': 95.159..."


### Map for Glucose and Ammonia

In [24]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Glucose"][ecoli_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/4092989759.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Glucose"][ecoli_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 93.8266666666663, 'ACONTb': 93.82666666666638, 'ICDHyr': 59.903703703704025, …

### Map for Glucose and Nitrate

In [25]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Glucose"][ecoli_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/2706785586.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Glucose"][ecoli_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 93.82666666666738, 'ACONTb': 93.82666666666736, 'ICDHyr': 59.90370370370317, …

### Map for Acetate and Ammonia

In [26]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Acetate"][ecoli_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/4019710256.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Acetate"][ecoli_blockage_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 95.16000000000092, 'ACONTb': 95.16000000000014, 'ICDHyr': 59.53333333333338, …

### Map for Acetate and Nitrate

In [27]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Acetate"][ecoli_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_47684/1584311455.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_blockage_df[ecoli_blockage_df["C_source"] == "Acetate"][ecoli_blockage_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 95.15999999999853, 'ACONTb': 95.15999999999852, 'ICDHyr': 59.533333333332, 'A…